# Lecture 5

In this lecture we’ll review some core Python patterns (loops and basic function patterns), then build up to a few “mind-bending” but extremely useful ideas:

- treating **functions as data** (passing/returning functions)
- using **`map`, `filter`, `reduce`**, and **comprehensions**
- understanding **iterators**, **generators**, and **lazy evaluation**
- using `*args` / `**kwargs` to **construct function calls**
- a final example: writing a small tool that **“vectorizes”** a function

The goal is not to memorize syntax—it’s to recognize *patterns* you can reuse.

A word on where this is going. We're used to thinking of data as numbers, and of
abstraction as something we do to numbers. In this lecture we start abstracting away
*code itself* — taking functions and algorithms and manipulating them, passing them
around, and using them as data. That's the idea I most want you to walk out with, and
it's the one I find genuinely a little mind-blowing.

The last thing we build is a piece of Python that takes a function and *vectorizes* it
for you: a function that only worked on single numbers now works on vectors, and you
didn't have to rewrite it. A function that takes a function and gives you back a new
function that does something new.


### Learning goals

By the end of this notebook you should be able to:

- write loops that iterate cleanly over sequences (and know when you need indices)
- recognize common “input → output” function patterns (number→list, list→number, list→list)
- explain what it means to pass a function as an argument and return a function
- describe (in words) what `map`, `filter`, and `reduce` do, and why they are often *lazy* in Python 3
- explain the difference between a **list** and an **iterator/generator**
- use `*` and `**` to unpack arguments when calling a function

Set your expectations correctly on this one. This lecture is closer to a **reference
manual** than to a story — it's a tour of the different ways Python lets you do these
things. I do not expect you to remember how to do every one of them. I expect you to
work through it once, understand *how each one works*, and have it in your head well
enough that when you need it later you know what to come back to and what you're
looking at when you see it in somebody else's code.

Work the practice problems scattered through the notebook on your own. That's not
busywork: if you can't do one, that's the signal that you haven't actually got the idea
yet, and it's much better to find that out here than later.


## Loops Patterns


Python `for`-loops are designed to iterate directly over **items**, not over indices.

That means the most common (and most readable) pattern is:

```python
for item in some_list:
    ...
```

You *can* loop over indices (e.g., `for i in range(len(lst)):`), but in Python you usually only do that when you truly need `i`.


In [1]:
for index in range(10):
    print(index)

0
1
2
3
4
5
6
7
8
9


#### A note about `range`

In **Python 3**, `range(10)` does **not** build a list in memory. It creates a lightweight *range object* that generates the numbers on demand.

Think about why this is the sensible design. If I'm counting from zero to a million, I
don't need to hold a million numbers in my head — when I'm at 9,000 I know perfectly
well what comes next. The range object works the same way: it doesn't store the
numbers, it just remembers where it is.

This is also your first encounter with a pattern you'll hit over and over in this
lecture: you call something, you expect a list, and you get back some object instead.
The fix is always the same one you see below — wrapping it in `list(...)` is you
*forcing* it to actually compute all the elements. Once you recognize that, a whole
class of confusing moments stops being confusing.

- You can loop over it directly (as above).
- If you want to *see* the values all at once, wrap it with `list(...)`:


In [2]:
list(range(10))

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

### Iterating over a list

Most of the time you should iterate over the **items** directly:

```python
for item in lst:
    ...
```

If you come from a “C-style” background, you may be used to writing:

```python
for i in range(len(lst)):
    item = lst[i]
```

That works in Python too, but it’s usually more verbose and easier to get wrong. Prefer the direct style unless you truly need the index.

It's worth seeing exactly what the C-style version is doing, because the difference is
the point. In that style you never really iterate over the list — you build a second
list of *indices*, count through those, and use each one to reach back into the real
list. Python skips all of that: you have a list of things, so take them one at a time.

If you learned to program somewhere else, you are probably writing loops the first way
out of habit. In Python you shouldn't. It's more to type, it's easier to get wrong, and
it's slower.


In [3]:
lst=['a','b','c']

# "Python style"
for item in lst:
    print(item)
    
# "C style"
for index in range(len(lst)):
    print(lst[index])

a
b
c
a
b
c


#### Checkpoint (try before peeking)

1. In Python 3, what does `range(5)` return: a list, or something else?
2. When is `for i in range(len(lst)):` actually the right tool?

<details>
<summary>Answers</summary>

1. A **`range` object** (an iterable), not a list. It *behaves* like a sequence of numbers, but it doesn’t store them all at once.  
2. When you truly need the **index** (e.g., to update `lst[i]`, align with another list/array, or write into a separate data structure).

</details>


### When you need the index: `enumerate`

If you need both the **index** and the **item**, use `enumerate`:

- it produces pairs like `(index, value)`
- you can unpack those pairs directly in the loop header

The reason this reads so cleanly is Python's assignment rules. `enumerate` hands you a
tuple each time around, and Python will unpack that tuple straight into two names in
the loop header. So you get the index and the item together, without ever managing a
counter yourself.

And note that `enumerate` behaves like `range` did — it doesn't do the enumeration
until you ask. Wrap it in `list(...)` if you want to see the pairs.


In [4]:
list(enumerate(lst))

[(0, 'a'), (1, 'b'), (2, 'c')]

**Next:** we’ll use `enumerate` to get both an index and a value as we loop.

In [5]:
for index,item in enumerate(lst):
    print(index,item)

0 a
1 b
2 c


#### Checkpoint (try before peeking)

1. What does `enumerate(lst)` produce?
2. How do you make `enumerate` start counting from 1 instead of 0?
3. Why is `enumerate` usually nicer than `range(len(lst))`?

<details>
<summary>Answers</summary>

1. An iterator of pairs like `(index, value)`.  
2. `enumerate(lst, start=1)`  
3. It’s clearer, less error‑prone, and you get both the index and the item directly.

</details>


### Iterating over multiple lists: `zip`

If you want to walk through multiple lists *in parallel*, use `zip`.

`zip(lst1, lst2)` pairs up items:

- first with first
- second with second
- and so on…

By default, `zip` stops when the **shortest** input runs out.

Ask yourself *why* you wanted the index in the first place. Almost always the answer is:
because I have two or three lists and I want the fifth element of each of them, then the
sixth element of each of them. That's the real need — and `zip` serves it directly, so
you don't need the index after all.

The name is the right mental picture: it interleaves the lists the way a zipper
interleaves two rows of teeth. And like the others, `zip` gives you an iterator, not a
list, until you ask for one.

So the whole first section reduces to three rules. Loop over items. If you think you
need the index, use `enumerate`. If the reason you wanted the index was to walk several
lists together, use `zip`.


In [6]:
lst1=['a','b','c']
lst2=['A','B','C']

for item1,item2 in zip(lst1,lst2):
    print(item1,item2)


a A
b B
c C


**Important:** in Python 3, many tools like `zip`, `map`, and `filter` return *iterators*.

That means they don’t immediately compute a full list of results. Instead, they produce values **one at a time** as you loop over them (this is called **lazy evaluation**). We’ll come back to why that matters later.


In [7]:
zip(lst1,lst2)

To make an iterator “do something”, you have to **consume** it by looping over it.

One easy way to force evaluation is to convert it to a list:

```python
list(zip(lst1, lst2))
```

(Just remember: building a list stores *all* results in memory. Iterators are useful when you don’t want to store everything at once.)


In [8]:
list(zip(lst1,lst2))

[('a', 'A'), ('b', 'B'), ('c', 'C')]

**Next:** we’ll use `zip` to iterate over multiple iterables in parallel.

In [9]:
list(zip(range(4),range(10)))

[(0, 0), (1, 1), (2, 2), (3, 3)]

**Next:** we’ll use `zip` to iterate over multiple iterables in parallel.

In [10]:
list(zip("Hello","World"))

[('H', 'W'), ('e', 'o'), ('l', 'r'), ('l', 'l'), ('o', 'd')]

#### Checkpoint (try before peeking)

1. What happens if you `zip` two lists of different lengths?
2. Why does `zip(lst1, lst2)` display something like `<zip object at ...>`?
3. How do you *see* all the paired results at once?

<details>
<summary>Answers</summary>

1. `zip` stops when the **shortest** input runs out.  
2. In Python 3, `zip` returns an **iterator** (lazy).  
3. Convert it: `list(zip(lst1, lst2))` (or loop over it).

</details>


### Practice: loop patterns (no index gymnastics)

Try these *without* using `range(len(...))`:

1. Given `lst = ['a', 'b', 'c']`, print each item on its own line.
2. Print **both** the index and the item in the format `0: a`, `1: b`, ...
3. Given `lower = ['a','b','c']` and `upper = ['A','B','C']`, print pairs like `a A` using `zip`.

<details>
<summary>One possible solution</summary>

```python
lst = ['a', 'b', 'c']

# 1) values only
for x in lst:
    print(x)

# 2) index + value
for i, x in enumerate(lst):
    print(str(i) + ": " + x)

# 3) pair two lists item-by-item
lower = ['a','b','c']
upper = ['A','B','C']
for lo, up in zip(lower, upper):
    print(lo, up)
```

</details>


## Basic Function Input/Output Patterns

A huge amount of programming comes down to a few repeatable patterns. Here are four you’ll see constantly:

1. **Number → List**: build up a list and return it  
2. **List → Number**: accumulate a single value (count, sum, max, …)  
3. **List → List (same length)**: transform each element (e.g., square every number)  
4. **List → List (shorter)**: filter down to a subset (e.g., only odds)

In each case, the structure is similar:

- create an output container (or accumulator)
- loop
- update the container
- `return` the result

Each of those needs its own particular pattern of looping, and you end up writing that
loop over and over. The move that matters — and it's the move the whole rest of this
lecture depends on — is to stop thinking "I write this loop" and start thinking "this is
the operation I'm doing." Once you can *name* the operation, you can find the tool that
performs it and stop writing the loop at all.


### Number → List

Example: return a list of **odd numbers below** `max_odd` (i.e., we generate candidates and keep the ones that match a condition).

Notice the shape, because we're about to see it four times: create a list, loop,
conditionally append to it, return it.

Two small notes on style while we're here. `num % 2` is the **mod** operator — it divides
and hands you the remainder — so `num % 2 == 1` is how you ask "is this odd?" And I'll
generally write `list()` rather than `[]` when I'm creating an empty list. It isn't more
correct, and I won't mark you down for the other one; it's just a bit more readable and
a bit less easy to misread, and it's what you'll see in a lot of real code.


In [11]:
def odds(max_odd):
    out_list=list()
    
    # Body
    for num in range(max_odd):
        if num%2==1:
            out_list.append(num)
    
    return out_list
        

**Next:** now that we've defined `odds()`, let's call it on a small example and inspect the result.

In [12]:
odds(13)

[1, 3, 5, 7, 9, 11]

### List to Number

Same skeleton, one change: what I'm accumulating is a single number rather than a list.
Start from a sensible default, loop, conditionally aggregate, return. It won't always be
a count — but you will always be starting somewhere and building something up.


In [13]:
def count_odds(lst):
    my_count=0
    
    # Body
    for num in lst:
        if num%2==1:
            my_count+=1  # my_count = my_count + 1
    
    return my_count

**Next:** now that we've defined `count_odds()`, let's call it on a small example and inspect the result.

In [14]:
count_odds([1,2,4,6,7,9])

3

### List to Same Length List

Again the same pattern. Create the output list, walk the input list, append one thing per
element. Because we append exactly once per input element, the output comes out the same
length as the input.


In [15]:
def square(lst):
    out_list = list()
    
    for num in lst:
        out_list.append(num*num)  
            
    return out_list

**Next:** now that we've defined `square()`, let's call it on a small example and inspect the result.

In [16]:
square([1,2,4,6,7,9])

[1, 4, 16, 36, 49, 81]

### List to Shorter List

And the filtering version, which is the same code with one difference: a conditional now
decides whether anything gets appended. That's it — that one `if` is the entire
difference between "same length out" and "shorter list out".

Four patterns, one skeleton. The code is simple, and there's nothing wrong with it. But
if we're writing it constantly, that's exactly the signal that we shouldn't have to.


In [17]:
def filter_odds(lst):
    out_list = list()
    
    for num in lst:
        if num%2==1:
            out_list.append(num)  
            
    return out_list

**Next:** now that we've defined `filter_odds()`, let's call it on a small example and inspect the result.

In [18]:
filter_odds([1,2,4,6,7,9])

[1, 7, 9]

### Practice: common function patterns

Write *short* functions that follow these patterns:

1. **Number → List:** `first_n_evens(n)` returns a list like `[2, 4, 6, ...]`.
2. **List → Number:** `count_negatives(xs)` returns how many items in `xs` are `< 0`.

<details>
<summary>One possible solution</summary>

```python
def first_n_evens(n):
    out = []
    k = 1
    while len(out) < n:
        out.append(2*k)
        k += 1
    return out

def count_negatives(xs):
    count = 0
    for x in xs:
        if x < 0:
            count += 1
    return count

print(first_n_evens(5))           # [2, 4, 6, 8, 10]
print(count_negatives([3,-1,0,-7]))  # 2
```

</details>


## An Example...

Let's stop listing patterns and build something with them. I want a small system for
multiplying numbers, vectors, and matrices, where everything happens element-wise and I
don't have to think about which case I'm in.

We'll do it in pieces: multiply a scalar by a scalar, a scalar by a vector, a vector by
a vector — and then a **dispatch function** that looks at what it was handed and calls
the right one. That dispatch is only possible because Python doesn't force us to declare
types up front: the same function can accept a number or a list and decide at the moment
of the call. That flexibility is one of the real powers of the language.

Then comes the trick. If, inside the list-multiply, I call `multiply` again instead of
using `*` directly, the whole thing becomes recursive — and it suddenly works on lists of
lists, and lists of lists of lists, to whatever depth you hand it. Matrices come out for
free. I didn't write any matrix code. I just let the abstraction call itself.

Functions that input/output lists:

In [19]:
def multiply_scalar_list(scalar,b):
    out = list()
    for item in b:
        out.append(scalar*item)
    return out

**Next:** now that we've defined `multiply_scalar_list()`, let's call it on a small example and inspect the result.

In [20]:
print(multiply_scalar_list(5,[1,2,3]))

[5, 10, 15]


**Next:** we’ll use `zip` to iterate over multiple iterables in parallel.

In [21]:
def multiply_lists(a,b):
    if len(a)!=len(b):
        print("Only can multiply lists of same length.")
        return None
    else:
        out = list()
        for item1,item2 in zip(a,b):
            out.append(item1*item2)
        return out 

**Next:** now that we've defined `multiply_lists()`, let's call it on a small example and inspect the result.

In [22]:
print(multiply_lists([1,2,3],[2,3,4]))
print(multiply_lists([1,2,3],[2,3,4,5]))

[2, 6, 12]
Only can multiply lists of same length.
None


We can combine the two functions and generalize: 

In [23]:
def multiply(a,b):
    if isinstance(a,(float,int)) and isinstance(b,(float,int)):
        return a*b
    elif isinstance(a,list) and isinstance(b,list):
        return multiply_lists(a,b)
    elif isinstance(a,list) and isinstance(b,(float,int)):
        return multiply_scalar_list(b,a)
    elif isinstance(b,list) and isinstance(a,(float,int)):
        return multiply_scalar_list(a,b)
    else:
        print("Invalid input.")
        return None

def multiply_lists(a,b):
    if len(a)!=len(b):
        print("Only can multiply lists of same length.")
        return None
    else:
        out = list()
        for item1,item2 in zip(a,b):
            # Note now we use multiply not * here:
            out.append(multiply(item1,item2))
        return out 
    
def multiply_scalar_list(scalar,b):
    out = list()
    for item in b:
        # Note now we use multiply not * here:
        out.append(multiply(scalar,item))
    return out

Note that the updated versions of `multiply_lists` and `multiply_scalar_list` re-use `multiply`, allowing further generalization.

In [24]:
print(multiply(2,2))

print(multiply(2,[1,2,3]))
print(multiply([1,2,3],2))

print(multiply([1,2,3],[2,3,4]))

print(multiply([[1,1,1],[2,2,2]], [[3,3,3],[4,4,4]]))


4
[2, 4, 6]
[2, 4, 6]
[2, 6, 12]
[[3, 3, 3], [8, 8, 8]]


### Practice: element‑wise list operations

1. Write `add_list_list(xs, ys)` that returns element‑wise sums (e.g., `[1,2] + [10,20] -> [11,22]`).
2. If the lists have different lengths, print an error message and return `None`.

<details>
<summary>One possible solution</summary>

```python
def add_list_list(xs, ys):
    if len(xs) != len(ys):
        print("Error: lists must have same length.")
        return
    out = []
    for x, y in zip(xs, ys):
        out.append(x + y)
    return out

print(add_list_list([1,2,3], [10,20,30]))  # [11, 22, 33]
```

</details>


### Functions as Arguments

Look again at that filter function and notice there are really *two* things tangled
together in it: the machinery that walks a list and keeps some items, and the test that
decides which ones. Those are separable.

If I pull the test out into its own function and pass it in, the walking machinery stops
being "the odd filter" and becomes "the filter", full stop. Want the even ones? Don't
touch the loop — hand it a different test. We've abstracted the *process* away from the
*condition*, and we could only do that because a function can be passed around exactly
like a number or a list.


In [25]:
def odd(num):
    return num%2==1

def filter_func(lst, func):
    out_list = list()
    
    for num in lst:
        if func(num):
            out_list.append(num)  
            
    return out_list


**Next:** we’ll keep only the items that pass a condition (filtering).

In [26]:
filter_func([1,2,4,6,7,9],odd)

[1, 7, 9]

**Next:** we’ll keep only the items that pass a condition (filtering).

In [27]:
def even(num):
    return num%2==0

filter_func([1,2,4,6,7,9],even)

[2, 4, 6]

`filter_func` takes:

- a list `lst`
- a **predicate** function `func` (a function that returns `True`/`False`)

It returns a *new list* containing only the elements of `lst` for which `func(element)` is `True`.


### Functions as Return

Now push it one step further. I don't want to keep handing the test in every time I call
it — I'd rather ask for an odd-filter *once* and get back a ready-made function.

So: a function whose argument is the test, and whose body defines and returns another
function. When you call the outer one, the test you passed gets captured, the inner
function is built with it baked in, and that inner function is handed back to you. What
you get is not a filtered list. It's a *filter* — a new function you can now call.

Read that slowly, because it's the hinge of the whole lecture. The inner function refers
to a name that lives in the enclosing function's context, and that context stays alive
and travels with it. You have wrapped the writing of code inside another piece of code —
you are writing code that writes code. In a lot of languages that's possible but awkward.
In Python it's natural.


In [28]:
def make_filter(func):

    def filter_func(lst):
        out_list = list()

        for num in lst:
            if func(num):
                out_list.append(num)  

        return out_list

    return filter_func


**Next:** now that we've defined `make_filter()`, let's call it on a small example and inspect the result.

In [29]:
filter_odd_0 = make_filter(odd)

**Next:** now that `filter_odd_0` is set up, we’ll use it in the following example.

In [30]:
type(filter_odd_0)

function

**Next:** we’ll keep only the items that pass a condition (filtering).

In [31]:
filter_odd_0([1,2,4,6,7,9])

[1, 7, 9]

`make_filter` is a **function factory**:

- it takes a predicate function `func`
- it *builds and returns* a new function that filters lists using `func`

This works because the inner function “remembers” (`closes over`) the value of `func` from the outer scope.


## Functions of Functions

I know this feels like a lot of ceremony for something you could have written directly.
Bear with it. I could just show you Python's built-in versions and move on, but I want to
go through it slowly and explicitly, so that when we use these things — and we're going
to use them constantly — you know exactly what's happening underneath rather than
treating it as magic.


In [32]:
def my_map(f,lst):
    out=list()
    for item in lst:
        out.append(f(item))
    return out

**Next:** now that we've defined `my_map()`, let's call it on a small example and inspect the result.

In [33]:
def square(x):
    return x*x

def cube(x):
    return x*x*x

print(my_map(square,[1,2,3]))
print(my_map(cube,[1,2,3]))

[1, 4, 9]
[1, 8, 27]


**Next:** we’ll apply a function to each item (mapping).

In [34]:
def operator(f):
    def my_map(lst):
        out=list()
        for item in lst:
            out.append(f(item))
        return out
    return my_map

**Next:** now that we've defined `operator()`, let's call it on a small example and inspect the result.

In [35]:
square_operator=operator(square)
cube_operator=operator(cube)

print(square_operator([1,2,3]))
print(cube_operator([1,2,3]))

[1, 4, 9]
[1, 8, 27]


### Practice: functions that return functions

1. Write `make_adder(k)` that returns a new function `add_k(x)` which returns `x + k`.
2. Use it to create `add10`, then apply `add10` to every element of `[1, 2, 3]`.

<details>
<summary>One possible solution</summary>

```python
def make_adder(k):
    def add_k(x):
        return x + k
    return add_k

add10 = make_adder(10)
print(add10(5))  # 15

xs = [1, 2, 3]
print([add10(x) for x in xs])  # [11, 12, 13]
```

</details>


## Lambda functions

Sometimes you need a tiny “one-off” function and it feels silly to write a full `def`.

A `lambda` creates a function *without giving it a name*:

```python
lambda x: x * x
```

A few notes:

- a `lambda` can take any number of arguments, but it must be **a single expression**
- the expression’s value is automatically returned
- if your logic needs multiple lines, `if`/`for` statements, or good readability, use `def` instead

Notice what prompted this. Look back at the last few sections: I keep writing tiny
one-line functions — square, cube, is-odd — purely so that I have something to pass in.
Up to now the only way to make a function was to define it and bind it to a name, and if
I'm never going to use that name again, the name is pure overhead. `lambda` is how you
say "I mean *this* function" without going through the ceremony of naming it.

It's worth being precise about the limitation, because it's not arbitrary. A lambda is
not a function in the full sense: it can't hold loops or statements, and it doesn't keep
state between calls. It evaluates to one thing. That's the whole trade — and for the
small tests and transformations we're passing around, it's exactly the right one.


In [36]:
square = lambda x: x * x

**Next:** now that `square` is set up, we’ll use it in the following example.

In [37]:
square(8)

64

**Next:** we’ll build on the previous cell and take the next step in the example.

In [38]:
def square(x):
    return x * x

To appreciate the power of `lambda`, let’s introduce a few built-in “functional programming” tools in Python:

- `map` (transform each element)
- `filter` (keep only elements that pass a test)
- `reduce` (combine a sequence into a single value)

We’ll also compare these to list comprehensions, which are often more readable in Python.


### map

`map(function, iterable)` applies `function` to each element of `iterable`.

In Python 3, `map(...)` returns an **iterator**, so you usually wrap it with `list(...)` when you want to see all results at once.

Now that you've written it yourself, let's use Python's. I had you build your own first
partly so you'd see there's nothing mysterious in it — and partly because the built-in
does the thing that surprises people.

Call `map` and you don't get a list. You get a **map object**: an iterator that hasn't
applied the function to anything yet, and won't until you ask.

That's **lazy evaluation**, and the reason for it is simple. I might describe a
computation over a million elements and then only ever look at the first five. Computing
the other 999,995 would be pure waste. So the language lets you *define* the work now and
*do* it only at the moment you need the answer. Wrap it in `list(...)` and you've asked
for all of it at once.


In [39]:
list1 = [1,2,3,4,5,6,7,8,9]

**Next:** now that `list1` is set up, we’ll use it in the following example.

In [40]:
eg = my_map(lambda x:x+2, list1)
print (eg)

[3, 4, 5, 6, 7, 8, 9, 10, 11]


**Next:** now that `eg` is set up, we’ll use it in the following example.

In [41]:
eg = map(lambda x:x+2, list1)
print (eg)

**Next:** now that `eg` is set up, we’ll use it in the following example.

In [42]:
list(eg)

[3, 4, 5, 6, 7, 8, 9, 10, 11]

**Next:** we’ll apply a function to each item (mapping).

In [43]:
def add_two(x):
    return x + 2

eg_0 = map(add_two, list1)
eg_0

**Next:** we’ll apply a function to each item (mapping).

In [44]:
# To see all results at once, convert to a list:
list(map(add_two, list1))

[3, 4, 5, 6, 7, 8, 9, 10, 11]

Because `map` is **lazy**, it only computes values when you *consume* the iterator (for example, by looping).

Also note: `map(...)` returns an **iterator**, which means it can be consumed only once. If you need the results multiple times, convert to a list or create a new `map` object.


In [45]:
eg_0 = map(add_two, list1)

for x in eg_0:
    print(x)

3
4
5
6
7
8
9
10
11


If you want to compute the result, just force it in the following way:

In [46]:
eg = list(map(lambda x:x+2, list1))
print (eg)

[3, 4, 5, 6, 7, 8, 9, 10, 11]


#### A very common gotcha: iterators are one‑pass

In Python 3, `map`, `filter`, and `zip` produce **iterators**. That means once you consume them (by looping or converting to a list), they’re **exhausted**.

The detail that catches people is what "exhausted" actually looks like: asking a spent
iterator for more gives you *nothing at all* — an empty result, not an error. So it fails
quietly. If you need the values twice, either keep the list you made the first time, or
build the iterator again from scratch.

Let’s see it happen:


In [47]:
it = map(lambda x: x + 1, [1, 2, 3])

list(it)   # consumes the iterator
list(it)   # empty now (already consumed)

[]

If you need the values more than once, either:

- store them in a list: `vals = list(map(...))`, or
- recreate the iterator (call `map(...)` again).


You can also add two lists.

In [48]:
list2 = [9,8,7,6,5,4,3,2,1]

**Next:** now that `list2` is set up, we’ll use it in the following example.

In [49]:
eg2 = list(map(lambda x,y:x+y, list1,list2))
print (eg2)

[10, 10, 10, 10, 10, 10, 10, 10, 10]


You can use `map` with `lambda`, but you can also pass any regular function (including built-ins like `str`).

In [50]:
eg3 = list(map(str,eg2))
print (eg3)

['10', '10', '10', '10', '10', '10', '10', '10', '10']


**Next:** we’ll apply a function to each item (mapping).

In [51]:
eg2 = list(map(lambda x,y:(x,y), list1,list2))
print (eg2)

[(1, 9), (2, 8), (3, 7), (4, 6), (5, 5), (6, 4), (7, 3), (8, 2), (9, 1)]


### filter

### `filter`

`filter(predicate, iterable)` keeps only the items for which `predicate(item)` is `True`.

In Python 3, `filter(...)` returns an **iterator** (not a list), so you often write:

```python
list(filter(...))
```

to see the results.

Nothing new here — it's the same shape as `map`, and it's the built-in version of the
filter you wrote by hand earlier.

One thing worth noticing about the function you pass it: where `map` wanted something that
turns a value into another value, `filter` wants something that turns a value into a
**boolean**. `lambda x: x < 5` is a perfectly ordinary expression; it just happens to
evaluate to True or False, and that's what makes it a predicate.


In [52]:
list1 = [1,2,3,4,5,6,7,8,9]

To get the elements which are less than 5,

In [53]:
list(filter(lambda x:x<5,list1))

[1, 2, 3, 4]

Notice what happens when `map()` is used.

In [54]:
list(map(lambda x:x<5, list1))

[True, True, True, True, False, False, False, False, False]

If the function you give to `map` returns `True`/`False`, then `map` produces a list of booleans.

`filter`, on the other hand, uses those booleans to decide which original elements to keep.


In [55]:
list(filter(lambda x:x%4==0,list1))

[4, 8]

### `reduce`

`reduce(function, iterable)` repeatedly combines items to produce a single result.

Conceptually, it does something like:

- combine the first two items → get an intermediate result  
- combine that result with the next item  
- repeat until the iterable is exhausted

`reduce` lives in `functools`, so you need to import it.

So we now have the third shape. `map` was list to list of the same length. `filter` was
list to shorter list. `reduce` is list to a *single value* — the last of the four patterns
we started with.

If you want to see what it's actually doing, replace the combining function with one that
builds a tuple instead of adding. The nesting that comes out shows you the order plainly:
it combined the first two, then combined *that* with the third, and so on. It looks a bit
like recursion running backwards.

And this is worth more than it may appear. Remember the checkers board — counting how many
pieces each player has is a loop we wrote out in full. That's a `reduce`, in one line. More
than that: this is how a great deal of large-scale data processing is organized. When work
has to be spread over a cluster or a supercomputer, it gets expressed in exactly these
terms — map, filter, reduce — precisely because operations in that form can be handed out
to many machines at once.


In [56]:
from functools import reduce

**Next:** with the imports ready, we’ll use them in the next step.

In [57]:
reduce(lambda x,y: x+y,[1,2,3])

6

**Next:** we’ll combine many items down to one result (reducing).

In [58]:
reduce(lambda x, y: (x,y), [1, 2, 3, 4, 5])

((((1, 2), 3), 4), 5)

### Practice: `map`, `filter`, and `reduce`

Let `xs = [1, 2, 3, 4, 5]`.

1. Use **`map`** to build a list of squares.
2. Use **`filter`** to keep only the even numbers.
3. Use **`reduce`** (from `functools`) to compute the sum.

<details>
<summary>One possible solution</summary>

```python
xs = [1, 2, 3, 4, 5]

squares = list(map(lambda x: x*x, xs))
evens   = list(filter(lambda x: x % 2 == 0, xs))

from functools import reduce
total = reduce(lambda a, b: a + b, xs)

print(squares)  # [1, 4, 9, 16, 25]
print(evens)    # [2, 4]
print(total)    # 15
```

</details>


### `functools.partial`: pre‑filling function arguments

A very common pattern is: *take an existing function* and create a **new** function that “locks in” some arguments.

This is similar to writing a small `lambda`, but `partial` can be clearer and gives the new function a nice `repr`.

We’ll use it again later when we talk about functions as data.

This one is here for reference more than for use. The pattern it serves is real — take an
existing function and pin down one of its arguments to make a new one — but it comes up
far less often than the tools above it. Know that it exists so you recognize it in
someone else's code.


In [59]:
from functools import partial

def power(base, exponent):
    return base ** exponent

square = partial(power, exponent=2)
cube   = partial(power, exponent=3)

print(square(5))  # 25
print(cube(2))    # 8

# Equivalent idea using lambda (sometimes fine, sometimes less readable):
square_lambda = lambda x: power(x, 2)
print(square_lambda(5))


25
8
25


## Shortcuts

Python has a few compact syntactic forms that can make code shorter. Use them when they improve readability (not just because they’re short).

The ternary is the one I actually use, and the place I use it is function arguments: I have
some option whose value depends on a condition, and I'd rather not break the flow with a
four-line `if` block just to set one name. Written inline it reads well and stays out of
the way.

Read it middle-out: the condition sits in the middle, the value for the true case on the
left, the value for the false case on the right. One caution — you *can* put a function
call in there, but if that function returns nothing (like `print`) then the whole
expression evaluates to `None`, which is almost never what you wanted.


In [60]:
if True:
    "True"
else:
    "False"

**Next:** we’ll build on the previous cell and take the next step in the example.

In [61]:
"True" if True else "False"

'True'

**Next:** we’ll build on the previous cell and take the next step in the example.

In [62]:
"True" if False else "False"

'False'

**Next:** we’ll build on the previous cell and take the next step in the example.

In [63]:
y = 15
x = 5 if y==15 else 13
print(x)

5


**Next:** we’ll build on the previous cell and take the next step in the example.

In [64]:
print("True") if True else print("False")

True


**Next:** we’ll build on the previous cell and take the next step in the example.

In [65]:
x = print("True") if True else print("False")
type(x)

True


NoneType

### List Comprehensions

As we have seen above, there is a common pattern where a function takes a list and returns another list of the same size. For example consider:

In [66]:
out = list()
for i in range(10):
    out.append(i)
out

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

We can do the same thing in a single line of code using list comprehensions:

In [67]:
out = [i*i for i in range(10)]
out

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]

Read the syntax as a sentence rather than as a loop. The opening bracket says you're
building a list. The first thing inside says what each element *is*. Then the `for` says
where the values come from — and it's the same `for` you'd have written anyway.

That's the mechanical translation: the body of the loop moves to the front, the loop
header moves to the back, and the `append` disappears because building the list is now
the bracket's job. What changes is how you think. You stop describing the *procedure* for
constructing a list and start describing *what the list is*.

These take a while to become natural, and that's normal. What usually converts people
isn't being persuaded that they're better — it's the tedium of writing the long version
one more time.


**Next:** we’ll build on the previous cell and take the next step in the example.

In [68]:
fruits = ["apple", "banana", "cherry", "kiwi", "mango"]
[x for x in fruits if "a" in x]

['apple', 'banana', 'mango']

**Next:** now that `fruits` is set up, we’ll use it in the following example.

In [69]:
list(filter(lambda x: "a" in x, fruits))

['apple', 'banana', 'mango']

#### Checkpoint (try before peeking)

1. In a list comprehension like `[f(x) for x in xs if cond(x)]`, what runs first: the `if` or `f(x)`?
2. How would you rewrite this comprehension as an equivalent `for` loop?

<details>
<summary>Answers</summary>

1. The `if` is a **filter**: only values that pass `cond(x)` get sent to `f(x)`.  
2. Create an empty list, loop over `xs`, check the condition, then append `f(x)`.

</details>


## Same task, three ways (loop vs comprehension vs `map`/`filter`)

Python gives you multiple ways to express the *same idea*. It’s worth seeing them side‑by‑side so you can choose what is clearest.

Task: from `0..9`, make a list of **squares of the even numbers**.


In [70]:
xs = list(range(10))

# 1) Plain loop (most explicit)
out1 = []
for x in xs:
    if x % 2 == 0:
        out1.append(x * x)

# 2) List comprehension (compact + common)
out2 = [x * x for x in xs if x % 2 == 0]

# 3) filter + map (functional style)
out3 = list(map(lambda x: x * x, filter(lambda x: x % 2 == 0, xs)))

out1, out2, out3

([0, 4, 16, 36, 64], [0, 4, 16, 36, 64], [0, 4, 16, 36, 64])

### Readability rules of thumb (worth memorizing)

Shorter code is **not automatically** better code.

A few guidelines that work well in practice:

- **Prefer the clearest version first.** A loop is often clearer than a “clever” one-liner.
- **Use list/dict comprehensions** when they stay readable in one (maybe two) lines.
- If your comprehension needs multiple `if`s, nested loops, or complex expressions, **switch back to a normal loop**.
- Don’t be afraid to write intermediate variables. Clarity beats cleverness.
- When you return to your own code a week later, you should still understand it quickly.

Python gives you powerful shortcuts — your job is to use them responsibly.


**Which should you use?**
- Use a **loop** when you need multiple steps, debugging prints, or clarity.
- Use a **comprehension** when it stays readable in one line.
- Use **`map`/`filter`** when it reads naturally *and* you’re comfortable with iterators.

(There isn’t one “correct” style — readability wins.)


### Dictionary Comprehensions

A dictionary is key-value pairs, so a dictionary comprehension needs both: curly braces
instead of square ones, and `key: value` where the single element used to go. The loop
part is unchanged.

Compare it with what we did last lecture, where we zipped two lists into pairs and passed
that to `dict(...)`. Both work, and it's worth seeing both — but this one reads better.

Using a similar syntax, we can quickly build dictionaries:

In [71]:
{i : chr(65+i) for i in range(4)}

{0: 'A', 1: 'B', 2: 'C', 3: 'D'}

**Next:** we’ll build on the previous cell and take the next step in the example.

In [72]:
[(i, chr(65+i)) for i in range(4)]

[(0, 'A'), (1, 'B'), (2, 'C'), (3, 'D')]

**Next:** we’ll build on the previous cell and take the next step in the example.

In [73]:
dict([(i, chr(65+i)) for i in range(4)])

{0: 'A', 1: 'B', 2: 'C', 3: 'D'}

### Practice: list & dictionary comprehensions

1. Build a list of squares for numbers `0..9`.
2. From `words = ["apple", "banana", "pear", "avocado"]`, keep only the words that contain the letter `"a"`.
3. Build a dictionary that maps each word to its length.

<details>
<summary>One possible solution</summary>

```python
squares = [i*i for i in range(10)]
print(squares)

words = ["apple", "banana", "pear", "avocado"]
with_a = [w for w in words if "a" in w]
print(with_a)

lengths = {w: len(w) for w in words}
print(lengths)
```

</details>


## Iterators, generators, and lazy evaluation

An **iterator** is an object you can repeatedly call `next(...)` on to get values *one at a time*.

- `iter(some_iterable)` gives you an iterator
- `next(iterator)` gives you the next value
- when the iterator is exhausted, it raises `StopIteration`

A key idea: **iterators are consumed**. Once you loop over them (or convert them to a list), they don’t “reset” automatically.

The mental model that makes iterators click: an iterator is a list **and** a position in
it, bundled together into one object. It doesn't own a copy of the data — it holds a
reference to it and remembers how far along it has got.

That bundling is what makes it useful. Because the position travels *with* the object, I
can hand an iterator to you and you can carry on from wherever I stopped. If what someone
needs is "the next thing", give them an iterator rather than a list and an index.

Consider the following:


In [74]:
iter_obj=iter([3,4,5,6,7,8,9])
next(iter_obj)

3

**Next:** now that `iter_obj` is set up, we’ll use it in the following example.

In [75]:
next(iter_obj)

4

**Next:** we’ll build on the previous cell and take the next step in the example.

In [76]:
list(iter_obj)

[5, 6, 7, 8, 9]

**Next:** we’ll build on the previous cell and take the next step in the example.

In [77]:
iter_obj = iter([3,4,5,6,7,8,9])

for i in iter_obj:
    print(i)

3
4
5
6
7
8
9


#### Checkpoint (try before peeking)

1. Why does `list(iter_obj)` return fewer values after you’ve already called `next(iter_obj)` a few times?
2. If you want to iterate twice, what should you do?

<details>
<summary>Answers</summary>

1. Because iterators don’t reset — you already consumed some values.  
2. Recreate the iterator (call `iter(...)` again), or convert to a list once and reuse the list.

</details>


### Generators and `yield`

A **generator function** looks like a normal function, but it uses `yield` instead of `return`.

- `return` ends the function immediately.
- `yield` *pauses* the function and hands back a value.
- The next time you ask for a value (with `next(...)` or a `for`-loop), the function resumes **right where it left off**, with all its local variables still in memory.

That’s why generators are great when the full result would be large—you can compute values **on demand** instead of building a huge list first.

Here's the part that makes generators different from anything you've written so far.
Normally when a function returns, it's over — its local variables are gone and its context
is dismantled. A generator's context *stays alive*. It hands you a value, freezes exactly
where it stands, and lets you go about your business. Ask for the next value and it thaws
out on the very next line with everything still in place.

Compare that with the iterator we just saw. An iterator remembers one thing: an index. A
generator remembers *whatever it takes* to compute the next value — the whole state of a
half-finished computation.

Each one you create is its own independent thing, too. Two generators from the same
function don't share anything; advancing one has no effect on the other.

Be honest with yourself about how much of this you need right now. You are unlikely to have
to *write* a generator in this course, and if you do, you'll look it up. What should stick
is how they work, because you use things built on them constantly. And you will meet them
for real later: in deep learning, training data is fed to the model by exactly this
mechanism — the training loop asks for the next batch, and a generator goes and fetches it.


In [78]:
def even_list(x):
    out = list()
    while(x!=0):
        if x%2==0:
            out.append(x)
        x-=1
    return out

def even_gen(x):
    while(x!=0):
        if x%2==0:
             yield x
        x-=1

**Next:** now that we've defined `even_list()`, let's call it on a small example and inspect the result.

In [79]:
even_list(10)

[10, 8, 6, 4, 2]

**Next:** we’ll build on the previous cell and take the next step in the example.

In [80]:
even_gen(10)

<generator object even_gen at 0x10a2e2440>

**Next:** we’ll build on the previous cell and take the next step in the example.

In [81]:
g=even_gen(10)
next(g),next(g)

(10, 8)

**Next:** we’ll build on the previous cell and take the next step in the example.

In [82]:
list(even_gen(10))

[10, 8, 6, 4, 2]

**Next:** we’ll build on the previous cell and take the next step in the example.

In [83]:
g=even_gen(10)
g2=even_gen(15)

next(g),next(g2)

(10, 14)

**Next:** now that `g` is set up, we’ll use it in the following example.

In [84]:
next(g)

8

**Next:** we’ll see how `yield` turns a function into a generator (lazy evaluation).

In [85]:
def prime_gen():
    """Generate prime numbers forever (simple, not optimized).

    This is mainly a demo of *generators that keep state* (the `primes` list)
    across many `yield` calls.
    """
    yield 2

    primes = [2]
    x = 3

    def is_prime(n):
        # Check divisibility by known primes up to sqrt(n)
        for p in primes:
            if p * p > n:
                break
            if n % p == 0:
                return False
        return True

    while True:
        if is_prime(x):
            primes.append(x)
            yield x
        x += 2  # only test odd candidates

**Next:** now that we've defined `prime_gen()`, let's call it on a small example and inspect the result.

In [86]:
g=prime_gen()
[ next(g) for _ in range(100)]

[2,
 3,
 5,
 7,
 11,
 13,
 17,
 19,
 23,
 29,
 31,
 37,
 41,
 43,
 47,
 53,
 59,
 61,
 67,
 71,
 73,
 79,
 83,
 89,
 97,
 101,
 103,
 107,
 109,
 113,
 127,
 131,
 137,
 139,
 149,
 151,
 157,
 163,
 167,
 173,
 179,
 181,
 191,
 193,
 197,
 199,
 211,
 223,
 227,
 229,
 233,
 239,
 241,
 251,
 257,
 263,
 269,
 271,
 277,
 281,
 283,
 293,
 307,
 311,
 313,
 317,
 331,
 337,
 347,
 349,
 353,
 359,
 367,
 373,
 379,
 383,
 389,
 397,
 401,
 409,
 419,
 421,
 431,
 433,
 439,
 443,
 449,
 457,
 461,
 463,
 467,
 479,
 487,
 491,
 499,
 503,
 509,
 521,
 523,
 541]

### Aside: the `_` convention

Notice the loop variable in the cell just above:

```python
[next(g) for _ in range(100)]
```

I wanted to run that loop a hundred times, but I never use the counter for anything.
Writing `_` instead of `i` is a convention that tells whoever reads the code "there is a
value here and I am deliberately ignoring it." It isn't special syntax — it's an ordinary
name, and it works exactly like `i` would. It just says something to the reader that `i`
doesn't.


### Generator comprehensions

A **generator comprehension** looks like a list comprehension, but uses parentheses `(...)` instead of brackets `[...]`.

- `[expr for x in ...]` builds the whole list immediately.
- `(expr for x in ...)` produces values **lazily**, one at a time.

This is a compact way to create a generator.

Same idea as a list comprehension, and the only change is the punctuation: round
parentheses instead of square brackets. Square brackets build the whole list right now.
Parentheses hand you a generator that will produce values as you ask for them.


In [87]:
gen_squares = (i * i for i in range(5))
gen_squares

<generator object <genexpr> at 0x10a5f5cb0>

**Next:** now that `gen_squares` is set up, we’ll use it in the following example.

In [88]:
next(gen_squares)

0

**Next:** we’ll build on the previous cell and take the next step in the example.

In [89]:
list(gen_squares)  # consume the rest

[1, 4, 9, 16]

**Next:** we’ll build on the previous cell and take the next step in the example.

In [90]:
# Once a generator is exhausted, it stays exhausted:
next(gen_squares, "done")

'done'

**Next:** we’ll build on the previous cell and take the next step in the example.

In [91]:
# Compare: list comprehension builds everything immediately
[i * i for i in range(5)]

[0, 1, 4, 9, 16]

**Next:** we’ll build on the previous cell and take the next step in the example.

In [92]:
gen_squares2 = (i * i for i in range(5))
type(gen_squares2), type([i * i for i in range(5)])

(generator, list)

### A quick tour of `itertools` (optional but very useful)

The `itertools` module is a “toolbox” for working with iterators and generators.

It’s especially useful when you want to:
- take the first *N* items from a generator,
- chain iterables together,
- work with infinite sequences safely (by slicing them).

A few favorites are `islice`, `chain`, and `count`.

`islice` is the one that earns its keep: it lets you slice an iterator the way you'd slice
a list. If you've got a generator producing primes forever and you want the 18th through
the 23rd, that's what you reach for — you can't index into an infinite sequence, but you
can slice it.

You will not need this module often, and when you do you will look up what's in it. The
reason to see it now is so you understand *why* it exists — ordinary list operations
don't apply to things that are produced on demand, so there has to be a parallel set of
tools for them.


In [93]:
import itertools as it

# islice: take the first N items from *any* iterator (even an infinite one)
print(list(it.islice(it.count(10, 10), 5)))  # 10, 20, 30, 40, 50

# chain: treat multiple iterables as one long iterable
print(list(it.chain([1, 2], [3, 4], "ab")))  # [1, 2, 3, 4, 'a', 'b']

# You can also use islice with your own generators.
g = prime_gen()
first_10_primes = list(it.islice(g, 10))
print(first_10_primes)


[10, 20, 30, 40, 50]
[1, 2, 3, 4, 'a', 'b']
[2, 3, 5, 7, 11, 13, 17, 19, 23, 29]


### Practice: using `itertools.islice`

1. Create an infinite iterator of even numbers using `itertools.count(0, 2)`.
2. Use `islice` to take the first 8 even numbers and turn them into a list.

<details>
<summary>One possible solution</summary>

```python
import itertools as it

evens = it.count(0, 2)              # 0, 2, 4, 6, ...
first8 = list(it.islice(evens, 8))  # take 8 of them
print(first8)
```

</details>


## Recursive functions

A recursive function calls **itself**. To be correct (and to terminate), it must have two parts:

- a **base case**: a condition where the function stops recursing and returns a result
- a **recursive case**: the function calls itself with a *smaller/simpler* input

When you design a recursive algorithm, make sure you can answer:

- What is the base case?
- What value should be returned in the base case?
- How do the arguments change in the recursive call?
- How are results combined as the recursion “unwinds” back to the original call?

(Also: Python has a recursion limit, so deep recursion can crash—iteration is often safer for large inputs.)

We've done recursion before, so this section is here mostly so it sits alongside the rest
of this material — it belongs with the abstraction ideas we've been building. Work through
the trace example below on your own; it's a chance to actually watch the calls stack up
and unwind, which is the part people find hard to picture.


#### When recursion is *not* the best choice (practical Python note)

Recursion can be elegant, but in Python it has a real limitation: **the recursion depth is limited** (to prevent crashing your program).

So for problems that might recurse “deeply” (hundreds/thousands of steps), an **iterative loop** is usually safer and often faster.

Example: Fibonacci numbers (iterative version):


In [94]:
import sys
sys.getrecursionlimit()

3000

**Next:** with the imports ready, we’ll use them in the next step.

In [95]:
def fib_iter(n):
    a, b = 0, 1
    out = []
    for _ in range(n):
        out.append(a)
        a, b = b, a + b
    return out

fib_iter(10)

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

#### Checkpoint (try before peeking)

1. What are the **two required pieces** of a correct recursive function?
2. What does it mean to “combine results as the recursion unwinds”?

<details>
<summary>Answers</summary>

1. A **base case** (stop) and a **recursive case** (call itself on a smaller/simpler input).  
2. Each call returns a partial result to its caller; the caller uses that to build the final answer.

</details>


In [96]:
def factorial(n):
    """Return n! (factorial) for n >= 0."""
    if n < 0:
        print("Error: factorial is not defined for negative numbers.")
        return
    if n in (0, 1):
        return 1
    return n * factorial(n - 1)

**Next:** now that we've defined `factorial()`, let's call it on a small example and inspect the result.

In [97]:
factorial(10)

3628800

**Next:** we’ll build on the previous cell and take the next step in the example.

In [98]:
def factorial_trace(n):
    """A version that shows the nested structure of the recursive calls."""
    if n < 0:
        print("Error: factorial is not defined for negative numbers.")
        return
    if n in (0, 1):
        return 1
    return (n, factorial_trace(n - 1))

**Next:** now that we've defined `factorial_trace()`, let's call it on a small example and inspect the result.

In [99]:
factorial_trace(10)

(10, (9, (8, (7, (6, (5, (4, (3, (2, 1)))))))))

**Next:** we’ll build on the previous cell and take the next step in the example.

In [100]:
def recur_fibo(n):
   if n <= 1:
       return n
   else:
       return(recur_fibo(n-1) + recur_fibo(n-2))

**Next:** now that we've defined `recur_fibo()`, let's call it on a small example and inspect the result.

In [101]:
recur_fibo(10)

55

**Next:** we’ll build on the previous cell and take the next step in the example.

In [102]:
[recur_fibo(i) for i in range(10)]

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

**Next:** we’ll build on the previous cell and take the next step in the example.

In [103]:
def rec_range(start, stop=None, step=1):
    """Recursive version of `range` (supports positive `step`)."""
    if stop is None:
        start, stop = 0, start

    if step <= 0:
        print("Error: this simple version only supports a positive step.")
        return

    if start >= stop:
        return []
    return [start] + rec_range(start + step, stop, step)

### Visual trace: watching recursion unwind

One reason recursion can feel “mysterious” is that you can’t *see* the nested calls.

A simple trick is to add a `depth` parameter and print with indentation.
Then you can watch the function call itself, hit the base case, and return values back up.



In [104]:
def sum_trace(xs, depth=0):
    indent = "  " * depth
    print(indent + "sum_trace(" + str(xs) + ")")

    # Base case
    if xs == []:
        print(indent + "=> 0")
        return 0

    # Recursive case
    result = xs[0] + sum_trace(xs[1:], depth + 1)
    print(indent + "=> " + str(result))
    return result

sum_trace([1, 2, 3])


sum_trace([1, 2, 3])
  sum_trace([2, 3])
    sum_trace([3])
      sum_trace([])
      => 0
    => 3
  => 5
=> 6


6

### Practice: write your own recursive list sum

Write a function `sum_list(xs)` that returns the sum of a list of numbers using recursion.

Hints:
- Base case: the empty list `[]`
- Recursive case: “first element + sum of the rest”

<details>
<summary>One possible solution</summary>

```python
def sum_list(xs):
    if xs == []:
        return 0
    return xs[0] + sum_list(xs[1:])

print(sum_list([1, 2, 3, 4]))  # 10
```

</details>


## Constructing Function Arguments

What `*` does is **de-reference** the list: it pulls the elements out and passes them one
by one as separate arguments. So `f(x)` hands the function a single argument that happens
to be a list, while `f(*x)` hands it as many arguments as there are elements. Try it with
`print` and you can see the difference immediately.

Dictionaries have the same idea but two versions, and the difference catches people. A
single `*` on a dictionary gives you its **keys** — occasionally what you want, usually
not. Double `**` gives you the **values**, passed as keyword arguments. And because they're
keyword arguments, the keys have to match the parameter names exactly, or you get an error.

The upside of that is order stops mattering. This is why it shows up everywhere in real
libraries: the tools you'll use have dozens of optional settings, and being able to keep
them in a dictionary — save it, load it, hand it around, pass it straight through to
another function — is far more manageable than tracking positional arguments. Going the
other way, `*args` in a definition means "I don't know how many arguments I'll be called
with"; they arrive as a list.

Imagine that you have a function that takes two arguments:

In [105]:
def f(one,two):
    print(one,two)

If you are in a situation where you have a list where the arguments are stored, you could call the function in this way:

In [106]:
x=[1,2]
f(x[0],x[1])

1 2


A better way is to **unpack** the list into positional arguments using `*`:

In [107]:
f(*x)

1 2


We can see what `*` does with the following example:

In [108]:
x=[1,2,3]
print(x)
print(*x)


[1, 2, 3]
1 2 3


You can do a similar thing with dictionaries:

In [109]:
y={"one":1,"two":2}
print(*y)
f(*y)

one two
one two


That isn’t quite right: iterating over a dictionary produces its **keys**.

If you want to pass the dictionary as **keyword arguments**, use `**` (and the keys must match the function’s parameter names):

In [110]:
f(**y)

1 2


Note that the expectation here is that the keys match the name of the arguments of the function. So the following doesn't work:

The cell below uses `try` / `except`, which we haven't covered and won't need again in this
lecture. All it's doing is catching the error and printing it, so the notebook keeps running
instead of stopping here — otherwise the mistake would halt everything below it and you
couldn't see the rest. Read the message it prints; that's the point of the cell.

We come back to `try` / `except` properly later in the course, when we start reading data
files and genuinely can't know ahead of time whether a value is a number or a string. That
turns out to be a good use for it in Python — but it's a different job from what we're doing
here, so for now we'll keep reporting problems the plain way, with a message and a return.

In [111]:
y = {"a": 1, "b": 2}

try:
    f(**y)
except TypeError as e:
    print("As expected, this fails because the dict keys don't match the function's parameter names:")
    print(" ", e)


As expected, this fails because the dict keys don't match the function's parameter names:
  f() got an unexpected keyword argument 'a'


#### Checkpoint (try before peeking)

1. What does `*xs` do in a function call like `f(*xs)`?
2. What does `**d` do in a call like `f(**d)`?
3. When would `**d` fail?

<details>
<summary>Answers</summary>

1. It **unpacks** a list/tuple into positional arguments.  
2. It **unpacks** a dict into keyword arguments.  
3. If the dict keys don’t match the function’s parameter names.

</details>


### Practice: unpacking with `*` and `**`

1. You have `vals = [3, 4]` and a function `add(a, b)` that returns `a + b`.  
   Call `add` using `vals` **without** writing `vals[0]` and `vals[1]`.
2. You have `params = {"sep": " | ", "end": " DONE\n"}`.  
   Use `print` with `**params` so the keyword arguments come from the dictionary.

<details>
<summary>One possible solution</summary>

```python
def add(a, b):
    return a + b

vals = [3, 4]
print(add(*vals))  # 7

params = {"sep": " | ", "end": " DONE\n"}
print("a", "b", "c", **params)
```

</details>


## Coding example

Let’s show off the power of Python with an example. In introductory physics you learn the 1‑D kinematics equations for constant acceleration:

- $x = x_0 + v_0 t + \tfrac{1}{2} a t^2$
- $v = v_0 + a t$

We can implement these equations directly as Python functions.

This is first-year physics, and the concrete version is worth holding on to: hold a ball
ten meters up, let go, and a second later it's at 5.1 meters, falling at 9.8 meters per
second.

But be clear about what changes when we write it down as code. On paper those are
*equations*: algebraic statements you can rearrange, solve for `t`, do symbolic work with.
In Python they become *assignments*: values come in, we compute, we bind the result to a
name and hand it back. It's a prescription for producing a number, not a symbolic object
you can manipulate. That distinction matters more than it looks.

Fair warning, too: this section is denser than what came before. The point isn't to
reproduce it from memory — it's to watch the pieces from this lecture combine into
something you couldn't have written an hour ago.


In [112]:
def x_a_t(a,t,x_0=0.,v_0=0.):
    x = x_0 + v_0 * t + 0.5 * a * t**2
    return x

**Next:** we’ll build on the previous cell and take the next step in the example.

In [113]:
def v_a_t(a,t,v_0=0.):
    v=v_0+a*t
    return v

So for example, the position and velocity of a rock dropped from 10 meters after 1 second is simply:

In [114]:
x_a_t(-9.8,1.,x_0=10.,v_0=0.)

5.1

**Next:** we’ll build on the previous cell and take the next step in the example.

In [115]:
v_a_t(-9.8,1.)

-9.8

In physics, 2‑D and 3‑D motion can often be treated as multiple independent 1‑D problems (one per coordinate).

Instead of rewriting the equations three times, we can write a **higher‑order function** that turns a scalar function into a “vectorized” function.

Assume vectors are stored as lists like `[x, y, z]`. Our vectorizer will:

1. take a scalar function $f_0$
2. create a new function that accepts the *same* arguments as $f_0$, but allows some arguments to be **lists**
3. “broadcast” any scalar arguments by repeating them to match the list length (for example, `t` might be a single time used for all coordinates)
4. call $f_0$ element‑by‑element and collect the results into an output list
5. return the new vectorized function


Let’s take this step by step.

First we need a way to determine the “vector length” we’re working with. We’ll look at all list‑valued arguments and find the maximum length:


In [116]:
args= [[1,2],[1,2,3],[1,2,3,4], 1]

max_len=0
for a in args:
    if isinstance(a,list):
        max_len=max(max_len,len(a))
    
print(max_len)


4


Here is a more compact way of doing the same thing using `filter` and `map`:

In [117]:
max_len = max(map(len,
                  filter(lambda x: isinstance(x,list),
                   args)))
print(max_len)

4


Next, we'll have to check that every argument is of the same length, and make lists out of ones that are not lists:

In [118]:
def create_new_args(args):
    """Normalize mixed scalar/list arguments.

    - If an argument is a list, it must have the same length as the longest list argument.
    - If an argument is a scalar, we *broadcast* it by repeating it to that same length.

    Returns a new list of list-arguments (all the same length).
    """

    list_args = [a for a in args if isinstance(a, list)]
    if not list_args:
        print("Error: at least one argument must be a list, so we know the target length.")
        return

    max_len = max(map(len, list_args))

    new_args = []
    for a in args:
        if isinstance(a, list):
            if len(a) != max_len:
                print("Error: all list arguments must have same length.")
                return
            new_args.append(a)
        else:
            new_args.append([a] * max_len)

    return new_args


Let’s test:

In [ ]:
# The list arguments have different lengths, so this reports the problem and returns None.
print(create_new_args([[1, 2], [1, 2, 3], 1]))


Error: all list arguments must have same length.
None


**Next:** we’ll build on the previous cell and take the next step in the example.

In [120]:
# This works: one list + one list + one scalar (scalar gets broadcast)
print(create_new_args([[1, 2], [3, 4], 5]))


[[1, 2], [3, 4], [5, 5]]


### Practice: can you rewrite `create_new_args` more compactly?

Goal: rewrite the core idea of `create_new_args` using **more functional / comprehension style**.

Try to do it in two steps:

1. Compute `max_len` from the list arguments.
2. Build `new_args` in a single expression (a list comprehension is a good fit).

Bonus: can you write a version that checks for mismatched list lengths in one line?

<details>
<summary>One possible solution (compact but still readable)</summary>

```python
def create_new_args_compact(args):
    list_args = [a for a in args if isinstance(a, list)]
    if not list_args:
        print("Error: at least one argument must be a list, so we know the target length.")
        return

    max_len = max(map(len, list_args))

    wrong_length = [a for a in args if isinstance(a, list) and len(a) != max_len]
    if len(wrong_length) > 0:
        print("Error: all list arguments must have same length.")
        return

    return [a if isinstance(a, list) else [a] * max_len for a in args]
```

</details>


In [121]:
def create_new_args_compact(args):
    """A compact version of create_new_args using comprehensions."""
    list_args = [a for a in args if isinstance(a, list)]
    if not list_args:
        print("Error: at least one argument must be a list, so we know the target length.")
        return

    max_len = max(map(len, list_args))

    wrong_length = [a for a in args if isinstance(a, list) and len(a) != max_len]
    if len(wrong_length) > 0:
        print("Error: all list arguments must have same length.")
        return

    return [a if isinstance(a, list) else [a] * max_len for a in args]


In [ ]:
def create_new_args_one_liner(args):
    return (lambda max_len:
                [ [a]*max_len if not isinstance(a, list) else
                  a if len(a) == max_len else None
                  for a in args ]
           )( max(map(len, filter(lambda x: isinstance(x, list), args))) )


Two things were wrong with the version above, and both are worth seeing because they are
easy mistakes to make once you start nesting these forms.

The first is a parsing problem. Writing `(lambda max_len: <expr> for a in args)` does *not*
give you a lambda whose body is a comprehension — Python reads it as a **generator
expression that produces lambdas**, and a generator isn't callable. The `for` has to sit
inside brackets in the lambda's body, so that the body is the list comprehension.

The second is simpler: `max_len` has to be a single number, but `len(args)*[max(...)]`
builds a *list* of that number repeated. Then `[a]*max_len` is list-times-list, which is
not a thing.

One honest limitation remains, and it's the reason `create_new_args_compact` above is the
version you'd actually use. A single expression has nowhere to put an error report — it
can't print a message and stop the way our other versions do. The best it can manage is to
*mark* the offending argument, which is what the `None` is doing. That's the real cost of
squeezing this into one line, and it's worth more than the line you save.


In [ ]:
print(create_new_args_one_liner([[1, 2], [3, 4], 5]))

# A mismatched list can only be marked, not rejected:
print(create_new_args_one_liner([[1, 2], [3, 4, 5], 5]))


[[1, 2], [3, 4], [5, 5]]
[None, [3, 4, 5], [5, 5, 5]]


**Next:** now that we've defined `create_new_args_compact()`, let's call it on a small example and inspect the result.

In [123]:
print(create_new_args_compact([[1, 2], [3, 4], 5]))

[[1, 2], [3, 4], [5, 5]]


**Next:** we’ll build on the previous cell and take the next step in the example.

In [ ]:
print(create_new_args_compact([[1, 2], [3, 4, 5], 5]))


Error: all list arguments must have same length.
None


### A quick connection to NumPy (preview)

The “vectorize” idea you’re about to see is closely related to what libraries like **NumPy** do all the time:

- You write math that looks like it works on scalars (single numbers)
- and NumPy applies it efficiently to whole arrays

**Important difference:** NumPy’s array operations run fast because the looping happens in optimized compiled code.  
Our pure‑Python “vectorize” is mainly for understanding the *idea*.

This isn't just an exercise, by the way. What we're about to build is a small version of
something NumPy does for real, and knowing that it's the same idea makes NumPy much less
mysterious when we get to it.

The reason NumPy is fast is that when you multiply two large arrays, the loop doesn't
happen in Python at all — you hand the whole operation to compiled libraries that do it
about as fast as it can be done. But that only covers the operations NumPy already knows.
Apply *your own* function element by element and you're back to writing the loop yourself,
which is exactly the problem we're solving here.


### Performance intuition (micro vs. macro)

You’ll sometimes hear advice like “list comprehensions are faster than for‑loops” or “`map` is faster”.

Sometimes that’s true, but the **big idea** is:

- micro-choices (loop vs comprehension vs `map`) often change speed by a *small factor*,
- while using the right algorithm or a vectorized library (like NumPy) can change speed by *orders of magnitude*.

Here’s a tiny timing demo. Don’t memorize the numbers — they change from machine to
machine and from run to run. Focus on the idea.



In [ ]:
import time

data = list(range(10_000))

start = time.time()
for repeat in range(200):
    out = []
    for x in data:
        out.append(x * x)
t_loop = time.time() - start

start = time.time()
for repeat in range(200):
    out = [x * x for x in data]
t_comp = time.time() - start

start = time.time()
for repeat in range(200):
    out = list(map(lambda x: x * x, data))
t_map = time.time() - start

print("for-loop:          ", round(t_loop, 3), "s")
print("list comprehension:", round(t_comp, 3), "s")
print("map + lambda:      ", round(t_map, 3), "s")


for-loop:           0.091 s
list comprehension: 0.042 s
map + lambda:       0.073 s


### Back to Vectorizing Functions

Let's lay out what has to happen. You give me a function that works on single numbers, and
several lists. I have to take the first element of each list and call your function with
them, then the second of each, and so on — and hand back the collected results. And I
don't want to hand back *results*; I want to hand back a **function** that produces them.

So there are three problems. Make all the arguments the same length — broadcasting a bare
number like 9.8 into a list, since if the acceleration is the same in every direction you
shouldn't have to type it three times, and raising an error on lists that genuinely
disagree. Then regroup: `create_new_args` gives me the arguments as a list *of* lists, and
what I need is the first element of each, then the second — which is what `zip` does, once
I remember to `*` the list so its elements become `zip`'s arguments rather than a single
argument. Then apply the function across those groups, which is a `map`.

Watch how many of the pieces from this lecture show up in the finding-the-maximum-length
step alone: filter the arguments down to the ones that are lists, map `len` over those,
take the max. The explicit loop right beside it does exactly the same job. Neither is
wrong. I'm showing you both on purpose, because you should be able to write either and
read both.


Finally we have to call a function on each element and store the results in a new list. We can use `zip` to simplify this operation. Here's an example of how `zip` works:

In [126]:
list(zip( [1,1,1,1], [2,2,2,2]))

[(1, 2), (1, 2), (1, 2), (1, 2)]

So for the output of `create_new_args` example above, it'll do the following, which is what we want:

In [127]:
list(zip([1, 2], [3, 4], [5, 5]))

[(1, 3, 5), (2, 4, 5)]

But the following won't work:

In [128]:
list(zip(create_new_args([[1,2],[3,4],5])))

[([1, 2],), ([3, 4],), ([5, 5],)]

Recall

In [129]:
create_new_args([[1,2],[3,4],5])

[[1, 2], [3, 4], [5, 5]]

We need to do:

In [130]:
list(zip(*create_new_args([[1,2],[3,4],5])))

[(1, 3, 5), (2, 4, 5)]

Back to calling a function on each element and store the results in a new list:

In [131]:
def apply_func(f,args):
    out=list()
    for new_args in zip(*args):
        out.append(f(*new_args))
    return out

Here is a fancier way to do the same thing:

In [132]:
def apply_func(f,args):
    return list(map(lambda x: f(*x),zip(*args)))

So putting it all together, here is the (x,y) location of an object dropped from (10,10) after 1 second:

In [133]:
apply_func(x_a_t,create_new_args([[-9.8,0],1,[10,10]]))

[5.1, 10.0]

We are not quite done yet… let’s pull all of this into a reusable function factory called `vectorize`:

In [134]:
def vectorize(f):
    def create_new_args(args):
        max_len = max(map(len,
                          filter(lambda x: isinstance(x,list),
                           args)))
        new_args=list()

        for a in args:
            if not isinstance(a,list):
                a0=[a]*max_len
            elif len(a)!=max_len:
                print("Error: all list arguments must have same length.")
                return
            else:
                a0=a
            new_args.append(a0)

        return new_args
    
    def apply_func(f,args):
        out=list()
        for new_args in zip(*args):
            out.append(f(*new_args))
        return out
    
    def vect_f(*args):
        return apply_func(f,create_new_args(args))
    
    return vect_f

Let's test:

In [135]:
vect_x_a_t=vectorize(x_a_t)
vect_x_a_t([-9.8,0],1,[10,10])

[5.1, 10.0]

Or simply:

In [136]:
vectorize(x_a_t)([-9.8,0],1,[10,10])

[5.1, 10.0]

Recall the earlier `multiply` example, we can almost recreate it:

In [137]:
multiply = vectorize(lambda x,y : x*y)

**Next:** now that `multiply` is set up, we’ll use it in the following example.

In [138]:
multiply(2,[1,2,3])

[2, 4, 6]

**Next:** we’ll build on the previous cell and take the next step in the example.

In [139]:
multiply([3,2,1],[1,2,3])

[3, 4, 3]

But not quite.

Why? Because Python’s `*` operator behaves differently depending on types:

- `3 * 4` is numeric multiplication
- `3 * [1, 2]` repeats the list (`[1, 2, 1, 2, 1, 2]`), which is **not** element‑wise scaling

To truly handle nested lists (matrices/tensors) element‑wise, we need recursion (like the earlier `multiply` example), or a library like NumPy.

Now we’ll also build a **recursive vectorize** that *does* handle nested lists element‑wise (see below).


#### Checkpoint (try before peeking)

1. In our `vectorize` helpers, why do we need `zip(*args)` (with a `*`) instead of just `zip(args)`?
2. What kind of bug happens if one list argument has a different length?

<details>
<summary>Answers</summary>

1. `zip(*args)` treats each list as a separate input, so it pairs up “first elements together, second elements together, …”.  
   `zip(args)` would instead zip a **single** list-of-lists, which is not what we want.  
2. Your element-wise pairing becomes inconsistent; the safest response is to report the problem and stop, rather than return a half-correct answer.

</details>


### Extension: a recursive `vectorize` that works on nested lists (matrices/tensors)

Our earlier `vectorize` handles scalars and 1‑D lists. We can push the idea further by making it **recursive**:

- If all arguments are scalars → call `f`
- If any argument is a list → broadcast scalars and apply element‑wise
- If list elements are themselves lists → recursion naturally handles deeper nesting

This is still a teaching tool (not optimized), but it demonstrates how element‑wise operations can scale from numbers → vectors → matrices.

This is the same recursive move we made at the very beginning with `multiply`, applied one
level up. This is also the point where it gets abstract enough that it's genuinely easy to
lose the thread, so don't be alarmed if you have to read it twice.

Take the conceptual point and don't worry about holding every line in your head: you can
write code whose job is to take code and turn it into other code. That's the power Python
hands you, and this whole lecture has been walking toward it.


In [140]:
def vectorize_recursive(f):
    def is_list(x):
        return isinstance(x, list)

    def vect(*args):
        # Base case: no argument is a list, so just call f
        list_args = [a for a in args if is_list(a)]
        if len(list_args) == 0:
            return f(*args)

        # Recursive case: element-wise over lists (with scalar broadcasting)
        n = len(list_args[0])
        wrong_length = [a for a in list_args if len(a) != n]
        if len(wrong_length) > 0:
            print("Error: all list arguments must have same length at each level.")
            return

        bargs = [a if is_list(a) else [a] * n for a in args]
        return [vect(*[a[i] for a in bargs]) for i in range(n)]

    return vect

**Next:** now that we've defined `vectorize_recursive()`, let's call it on a small example and inspect the result.

In [141]:
mul_elemwise = vectorize_recursive(lambda x, y: x * y)

mul_elemwise(3, [1, 2, 3]), mul_elemwise([1, 2, 3], [10, 20, 30])

([3, 6, 9], [10, 40, 90])

**Next:** now that `mul_elemwise` is set up, we’ll use it in the following example.

In [142]:
# Works on nested lists too:
mul_elemwise(3, [[3, 2, 1], [1, 2, 3]])

[[9, 6, 3], [3, 6, 9]]

**Next:** we’ll build on the previous cell and take the next step in the example.

In [143]:
multiply(3,[[3,2,1],[1,2,3]])

[[3, 2, 1, 3, 2, 1, 3, 2, 1], [1, 2, 3, 1, 2, 3, 1, 2, 3]]

## Summary

Key takeaways from this lecture:

- **Looping:** iterate over *items* directly; use `enumerate` when you need indices, and `zip` when you need to iterate over multiple sequences together.
- **Common function patterns:** build and return a list; accumulate into a single value; transform a list; filter a list.
- **Functions are values:** you can pass functions as arguments and return functions (this is the foundation of `map`/`filter` and many powerful abstractions).
- **`lambda`:** convenient for small, one‑line functions; use `def` for anything more complex.
- **Functional tools:** `map`, `filter`, and `reduce` can express common patterns concisely—but in Python 3 they often return **iterators**, so they’re evaluated lazily.
- **Iterators & generators:** iterators produce values on demand; generators use `yield` to pause and resume, keeping local state alive between values.
- **Recursion:** always identify the base case and how the recursive case makes progress toward it.
- **Argument unpacking:** `*args` unpacks positional arguments, `**kwargs` unpacks keyword arguments—useful when writing wrapper/helper functions.
- **Vectorizing functions:** the final example combines these ideas into “code that writes code”—turning a scalar function into a function that works over lists.

### Optional practice
- Rewrite one loop in this notebook using (a) a list comprehension and (b) `map`/`filter`. Compare readability.
- Extend `vectorize` so it can handle **nested lists** (hint: recursion).
- Write a generator that yields the Fibonacci sequence efficiently (without the slow recursive definition).

That's a lot of ground, and I want to be clear about what I expect. Not that you can
reproduce any of it from memory — that I expect you to have gone through it once,
understood how each piece works, and know it well enough to come back to when you need it.
These ideas take time and real use to sink in. This is your first pass, not your last.

If there's one thing to carry forward, it's the shift in level. We began by asking how to
write a loop. We ended by writing a function that manufactures functions. Along the way the
question stopped being "what steps do I perform?" and became "what operation am I doing?" —
and once you can name the operation, you can usually stop writing the steps.
